# S&P 500 options — premium ranking dashboard
## Bloomberg BQuant · selected stocks · strike ranges · puts and calls

Upload this notebook to your Bloomberg **BQuant** project and choose **Run All**.
The initial request is only for one stock. No separate `.py` file is required.

**New in v2:** custom constituent lists; optional stock last-price range; absolute or percentage-of-spot strike ranges; highest-premium put/call rankings; normalized premium ratios; one contract per stock/side; and a direct `dashboard.screen(...)` function.

**Default definition of “expensive”: highest selected premium**, using LAST unless changed.
This is not an implied-volatility or fair-value ranking. No trades are placed.

### Example dashboard settings
Choose **Selected stocks**, enter `AAPL, MSFT, NVDA`, and select **Strike as % of spot**,
minimum `90`, maximum `100`. Select **DTE range**, `20` to `45`, and keep
**All within range**. Click **Load / rebuild**. In **Premium ranking**, choose
**Puts**, **Last**, **Premium (highest first)**, and Top N `50`.

Use **Absolute strike** and bounds `100`–`200` for native strike values instead.
Use **Exact date** to compare the same expiration across stocks.
Choose **Calls** or **Both** to change sides without another Bloomberg request.

**Validation boundary:** offline tests cover Python logic and mocked integration, not live Bloomberg permissions,
field availability, or the BQuant frontend. Prices are snapshots and may be stale, delayed or unavailable.


## 1 · Imports and environment
Use Bloomberg-provided `bql`, pandas, numpy, ipywidgets and IPython. No pip-install cells are needed. Native BQL field names remain isolated in `FIELD_NAMES`.

In [ ]:
# 1. Imports and configuration
from concurrent.futures import ThreadPoolExecutor
from datetime import timedelta
from pathlib import Path
import html
import math
import re
import warnings
import threading
import traceback

import numpy as np
import pandas as pd
try:
    import bql
    import ipywidgets as widgets
    from IPython.display import display, FileLink, clear_output
except ImportError as exc:
    raise ImportError(
        "Run this notebook in a Bloomberg BQuant Python environment with "
        "Bloomberg's bql and ipywidgets packages enabled. Do not install the "
        "unrelated public PyPI package named 'bql'."
    ) from exc

INDEX = "SPX Index"
DEFAULT_STOCK = "AAPL US Equity"
MAX_WORKERS = 1             # Serial by default. Optional: 2-4 independent workers.
QUOTE_BATCH_SIZE = 200      # Number of selected option contracts per quote request.
PAGE_SIZE = 100             # Display paging only; does not truncate the export.
AUTO_LOAD_SINGLE = True     # Never automatically starts an index-wide scan.

# Native BQL data items (not Excel/BDP OPT_* field mnemonics).
# If Bloomberg changes a field in your environment, this is the one place to edit.
FIELD_NAMES = {
    "EXPIRY": "expire_dt", "STRIKE": "strike_px", "SIDE": "put_call",
    "LAST": "px_last", "BID": "px_bid", "ASK": "px_ask",
    "NAME": "name", "STOCK_LAST": "px_last",
}


def utc_now():
    """Application retrieval time, NEVER represented as an exchange quote time."""
    return pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds")


def market_date():
    """Use New York's calendar date rather than the notebook server's timezone."""
    return pd.Timestamp.now(tz="America/New_York").date()


def data_items(service, names):
    return {name: getattr(service.data, FIELD_NAMES[name])() for name in names}

## 2 · BQL adapter and bounded chain discovery
The stock and expiry/strike constraints are applied before requesting option quotes. All listed strikes inside the selected range are used by default.

In [ ]:
# 2. Bloomberg query adapter and contract selection

def response_frame(response, aliases):
    """Join one current observation per ID; keep each item's DATE / CURRENCY.

    Joining by security ID is deliberate. Joining quotes and reference fields
    on DATE could lose contracts, and pivoting with mean could hide duplicates.
    Unexpected multi-row observations raise an error rather than silently choosing.
    """
    items = list(response)
    aliases = list(aliases)
    if len(items) != len(aliases):
        raise ValueError("BQL did not return one response item for every requested field.")
    parts = []
    for item, alias in zip(items, aliases):
        f = item.df().copy()
        if "ID" not in f.columns:
            f = f.reset_index()
        if "ID" not in f.columns:
            raise ValueError("BQL response has no ID column for " + alias)
        if alias not in f.columns:
            matches = [c for c in f.columns if str(c).upper() == alias.upper()]
            if len(matches) != 1:
                raise ValueError("Unexpected BQL columns for %s: %s" % (alias, list(f.columns)))
            f = f.rename(columns={matches[0]: alias})
        # Do not allow extra dimensions to be silently collapsed.
        if f["ID"].duplicated().any():
            raise ValueError("Multiple observations per ID for %s. Expected a snapshot, not a time series." % alias)
        kept = ["ID", alias]
        renames = {}
        for meta in ("DATE", "CURRENCY", "AS_OF_DATE"):
            if meta in f.columns:
                kept.append(meta)
                renames[meta] = alias + "_" + meta
        f = f[kept].rename(columns=renames)
        f["ID"] = f["ID"].astype(str)
        parts.append(f.set_index("ID"))
    if not parts:
        return pd.DataFrame(index=pd.Index([], name="ID"))
    return pd.concat(parts, axis=1, join="outer")


def execute_snapshot(service, universe, fields):
    # No historical date range, forward fill, or explicit cached-mode request.
    request = bql.Request(universe, fields)
    return response_frame(service.execute(request), fields.keys())


def load_members(service):
    f = execute_snapshot(service, service.univ.members(INDEX),
                         data_items(service, ["NAME", "STOCK_LAST"]))
    if f.empty:
        raise ValueError("No constituents returned for " + INDEX)
    f.index.name = "TICKER"
    f["NAME"] = f["NAME"].fillna("").astype(str)
    f["STOCK_LAST"] = pd.to_numeric(f["STOCK_LAST"], errors="coerce")
    f["STOCK_FETCHED_UTC"] = utc_now()
    return f.sort_index()


def date_window(expiry_mode, exact, target_dte, window_days, today=None):
    today = today or market_date()
    target = today + timedelta(days=int(target_dte))
    if expiry_mode == "Exact date":
        if exact is None:
            raise ValueError("Select an exact expiration date.")
        if exact < today:
            raise ValueError("Expiration must not be earlier than today's New York date.")
        return exact, exact, exact
    return (max(today, target - timedelta(days=int(window_days))),
            target + timedelta(days=int(window_days)), target)


def select_contracts(frame, spot, start, end, target, mode, band_pct, strike_count,
                     strike_range=None, strike_mode="PCT_SPOT"):
    """Select actual listed expiries and strikes; no theoretical interpolation."""
    f = frame.copy()
    f["EXPIRY"] = pd.to_datetime(f["EXPIRY"], errors="coerce").dt.normalize()
    f["STRIKE"] = pd.to_numeric(f["STRIKE"], errors="coerce")
    f["SIDE"] = (f["SIDE"].astype(str).str.strip().str.upper()
                 .replace({"C": "CALL", "P": "PUT"}))
    structurally_valid = (f["EXPIRY"].notna() & np.isfinite(f["STRIKE"])
                          & f["STRIKE"].gt(0) & f["SIDE"].isin(["CALL", "PUT"]))
    bad_rows = int((~structurally_valid).sum())
    f = f.loc[structurally_valid].copy()
    bounds = strike_range if strike_range is not None else (100 - band_pct, 100 + band_pct)
    lower, upper = strike_bounds(spot, bounds, strike_mode)
    f = f[(f["EXPIRY"] >= pd.Timestamp(start)) & (f["EXPIRY"] <= pd.Timestamp(end))
          & f["STRIKE"].between(lower, upper)]
    if f.empty:
        return f, bad_rows
    if mode == "Nearest target DTE":
        expires = list(f["EXPIRY"].unique())
        chosen = min(expires, key=lambda d: (abs(pd.Timestamp(d) - pd.Timestamp(target)), pd.Timestamp(d)))
        f = f.loc[f["EXPIRY"] == chosen]
    if strike_count:  # 0 means all listed strikes inside the selected band.
        blocks = []
        for _, block in f.groupby("EXPIRY", sort=True):
            strikes = sorted(block["STRIKE"].unique(), key=lambda k: (abs(k - spot), k))
            blocks.append(block[block["STRIKE"].isin(strikes[:int(strike_count)])])
        f = pd.concat(blocks) if blocks else f.iloc[:0]
    return f.sort_values(["EXPIRY", "STRIKE", "SIDE"]), bad_rows


_thread_state = threading.local()


def discover_stock(ticker, member, config):
    """One bounded metadata request per stock, with a separate service per worker."""
    spot = pd.to_numeric(pd.Series([member.get("STOCK_LAST")]), errors="coerce").iloc[0]
    if not np.isfinite(spot) or spot <= 0:
        return pd.DataFrame(), "Missing/nonpositive stock price", ""
    try:
        if not hasattr(_thread_state, "service"):
            _thread_state.service = bql.Service()
        svc = _thread_state.service
        expiry = getattr(svc.data, FIELD_NAMES["EXPIRY"])()
        strike = getattr(svc.data, FIELD_NAMES["STRIKE"])()
        # These bounds are applied on Bloomberg's side, before downloading chains.
        condition = svc.func.and_(expiry >= config["start"].isoformat(),
                                  expiry <= config["end"].isoformat())
        lower, upper = strike_bounds(spot, config["strike_range"], config["strike_mode"])
        condition = condition.and_(strike >= lower)
        condition = condition.and_(strike <= upper)
        universe = svc.univ.filter(svc.univ.options(ticker), condition)
        f = execute_snapshot(svc, universe, data_items(svc, ["EXPIRY", "STRIKE", "SIDE"]))
        f, invalid = select_contracts(f, spot, config["start"], config["end"],
                                      config["target"], config["expiry_mode"],
                                      config.get("band", 20), config["strikes"],
                                      config["strike_range"], config["strike_mode"])
        if f.empty:
            return f, "No matching contracts", ("%d invalid metadata rows omitted" % invalid if invalid else "")
        f = f.reset_index()
        f["TICKER"] = ticker
        f["NAME"] = member.get("NAME", "")
        f["STOCK_LAST"] = spot
        f["STOCK_LAST_DATE"] = member.get("STOCK_LAST_DATE", pd.NaT)
        f["STOCK_CURRENCY"] = member.get("STOCK_LAST_CURRENCY", "")
        f["STOCK_FETCHED_UTC"] = member.get("STOCK_FETCHED_UTC", "")
        return f, "Selected", ("%d invalid metadata rows omitted" % invalid if invalid else "")
    except Exception as exc:
        return pd.DataFrame(), "Metadata error", "%s: %s" % (type(exc).__name__, exc)

## 3 · Prices, quality checks and paired views
LAST, BID, ASK and calculated MID remain separate; unavailable prices are not zero-filled. Full option IDs are retained.

In [ ]:
# 3. Quote retrieval, quality checks, and call/put pairing

def fetch_quotes(service, metadata, progress=None):
    """Refresh only the selected contracts. Failed fields remain missing."""
    ids = metadata["ID"].drop_duplicates().tolist()
    blocks, errors = [], []
    for offset in range(0, len(ids), QUOTE_BATCH_SIZE):
        batch = ids[offset:offset + QUOTE_BATCH_SIZE]
        fields = data_items(service, ["LAST", "BID", "ASK"])
        try:
            quotes = execute_snapshot(service, batch, fields)
        except Exception as bulk_exc:
            # A missing/unauthorized bid field should not suppress an available last.
            parts = []
            for alias, expression in fields.items():
                try:
                    parts.append(execute_snapshot(service, batch, {alias: expression}))
                except Exception as exc:
                    errors.append({"STAGE": "Quotes", "SCOPE": "%d contracts, first: %s" % (len(batch), batch[0]),
                                   "FIELD": alias, "ERROR": "%s: %s" % (type(exc).__name__, exc)})
            quotes = pd.concat(parts, axis=1) if parts else pd.DataFrame(index=batch)
            if parts and not any(e["SCOPE"].endswith(batch[0]) for e in errors):
                errors.append({"STAGE": "Quotes", "SCOPE": batch[0], "FIELD": "Combined request",
                               "ERROR": "Recovered using separate fields: " + str(bulk_exc)})
        quotes = quotes.reindex(batch)
        quotes.index.name = "ID"
        quotes["QUOTES_FETCHED_UTC"] = utc_now()
        blocks.append(quotes)
        if progress:
            progress(min(offset + len(batch), len(ids)), len(ids))
    quotes = pd.concat(blocks) if blocks else pd.DataFrame(index=pd.Index([], name="ID"))
    return metadata.merge(quotes.reset_index(), on="ID", how="left", validate="many_to_one"), errors


def enrich_quotes(raw, today=None):
    f = raw.copy()
    for col in ("LAST", "BID", "ASK"):
        if col not in f:
            f[col] = np.nan
        f[col] = pd.to_numeric(f[col], errors="coerce")
        for suffix in ("DATE", "CURRENCY"):
            key = col + "_" + suffix
            if key not in f:
                f[key] = pd.NaT if suffix == "DATE" else ""
    bid, ask = f["BID"], f["ASK"]
    bid_date = pd.to_datetime(f["BID_DATE"], errors="coerce", utc=True).dt.normalize()
    ask_date = pd.to_datetime(f["ASK_DATE"], errors="coerce", utc=True).dt.normalize()
    date_mismatch = bid_date.notna() & ask_date.notna() & bid_date.ne(ask_date)
    bc = f["BID_CURRENCY"].fillna("").astype(str)
    ac = f["ASK_CURRENCY"].fillna("").astype(str)
    currency_mismatch = bc.ne("") & ac.ne("") & bc.ne(ac)
    missing = ~np.isfinite(bid) | ~np.isfinite(ask)
    invalid = (bid < 0) | (ask <= 0) | (bid > ask)
    good = ~(missing | invalid | date_mismatch | currency_mismatch)
    f["MID"] = ((bid + ask) / 2).where(good)
    f["QUOTE_NOTE"] = np.select(
        [missing, bid.lt(0) | ask.le(0), bid.gt(ask), date_mismatch, currency_mismatch],
        ["Missing bid/ask", "Negative bid or nonpositive ask", "Crossed bid/ask",
         "Bid/ask observation dates differ", "Bid/ask currencies differ"],
        default="Indicative quote; intraday age unverified")
    f["DTE"] = (pd.to_datetime(f["EXPIRY"]) - pd.Timestamp(today or market_date())).dt.days
    spot = pd.to_numeric(f["STOCK_LAST"], errors="coerce")
    f["STRIKE_VS_SPOT_PCT"] = 100 * (f["STRIKE"] / spot.where(spot > 0) - 1)
    return f


def pair_contracts(raw):
    """One row per stock/expiry/strike. Never average ambiguous option series.

    More than one contract on either side makes BOTH paired prices blank. All
    original contracts and quotes remain available in the individual-contract view.
    Deliverables are not verified; use full IDs to inspect nonstandard contracts.
    """
    if raw.empty:
        return pd.DataFrame()
    keys = ["TICKER", "EXPIRY", "STRIKE"]
    quote_cols = ["LAST", "BID", "ASK", "MID", "LAST_DATE", "BID_DATE", "ASK_DATE",
                  "LAST_CURRENCY", "BID_CURRENCY", "ASK_CURRENCY", "QUOTES_FETCHED_UTC", "QUOTE_NOTE"]
    rows = []
    for (ticker, expiry, strike), block in raw.groupby(keys, sort=True):
        first = block.iloc[0]
        row = {"TICKER": ticker, "NAME": first["NAME"], "STOCK_LAST": first["STOCK_LAST"],
               "STOCK_LAST_DATE": first["STOCK_LAST_DATE"], "STOCK_CURRENCY": first["STOCK_CURRENCY"],
               "STOCK_FETCHED_UTC": first["STOCK_FETCHED_UTC"], "EXPIRY": expiry, "DTE": first["DTE"],
               "STRIKE": strike, "STRIKE_VS_SPOT_PCT": first["STRIKE_VS_SPOT_PCT"]}
        counts = block.groupby("SIDE").size()
        ambiguous = bool((counts > 1).any())
        row["PAIR_NOTE"] = ("Multiple series: inspect individual contracts" if ambiguous else
                            "Call or put absent" if len(counts) < 2 else "")
        for side in ("CALL", "PUT"):
            leg = block[block["SIDE"] == side]
            row[side + "_COUNT"] = len(leg)
            row[side + "_ID"] = " | ".join(leg["ID"].astype(str))
            for col in quote_cols:
                row[side + "_" + col] = (leg.iloc[0][col] if len(leg) == 1 and not ambiguous else
                                          np.nan if col in ("LAST", "BID", "ASK", "MID") else "")
        rows.append(row)
    return pd.DataFrame(rows)

## 4 · Reusable filtering and ranking
`rank_options(raw, ...)` operates only on already loaded contracts. Relative strike bounds use **100 × K / stock last**; `90` to `100` means 90%–100% of spot. All bounds are inclusive.

In [ ]:
# 3b. Stock/range validation and reusable option-premium ranking

def checked_range(values, name, positive=False):
    """Validate a two-element inclusive numeric range."""
    if values is None:
        return None
    if isinstance(values, str) or len(values) != 2:
        raise ValueError(name + " must contain exactly (minimum, maximum).")
    try:
        lo, hi = float(values[0]), float(values[1])
    except (TypeError, ValueError) as exc:
        raise ValueError(name + " must contain two numbers.") from exc
    if not np.isfinite([lo, hi]).all() or lo > hi:
        raise ValueError(name + " must be finite, with minimum <= maximum.")
    if positive and lo <= 0:
        raise ValueError(name + " minimum must be greater than zero.")
    return lo, hi


def resolve_stock_selection(stocks, available):
    """Resolve exact Bloomberg IDs or unique short symbols. Never guess a ticker.

    String lists accept comma, semicolon or newline separators. A string of bare
    symbols separated by spaces also works. Full Bloomberg IDs must be separated
    by commas/newlines. Every requested stock must be in `available`.
    """
    ids = list(dict.fromkeys(str(x) for x in available))
    if stocks is None:
        return ids
    if isinstance(stocks, str):
        text = stocks.strip()
        if text.upper() in ("SPX", "SPX INDEX", "ALL S&P 500", "ALL"):
            return ids
        parts = [x.strip() for x in re.split(r"[,;\n]+", text) if x.strip()]
        if len(parts) == 1 and not re.search(r"\bEQUITY\b", text, re.I):
            parts = text.split()
    else:
        parts = [str(x).strip() for x in stocks if str(x).strip()]
    if not parts:
        raise ValueError("Enter at least one stock, for example AAPL, MSFT, NVDA.")
    full = {x.upper(): x for x in ids}
    short = {}
    for identifier in ids:
        short.setdefault(identifier.split()[0].upper(), []).append(identifier)
    selected, invalid = [], []
    for part in parts:
        key = part.upper()
        matches = [full[key]] if key in full else short.get(key, [])
        if len(matches) != 1:
            invalid.append(part + (" (ambiguous)" if len(matches) > 1 else " (not in this universe)"))
        elif matches[0] not in selected:
            selected.append(matches[0])
    if invalid:
        raise ValueError("Unresolved stocks: " + ", ".join(invalid) +
                         ". Use an exact Bloomberg ID from dashboard.members.index.")
    return selected


def strike_bounds(spot, strike_range, strike_mode="PCT_SPOT"):
    """ABSOLUTE uses native strike units; PCT_SPOT uses 100 * strike / stock last."""
    lo, hi = checked_range(strike_range, "Strike range", positive=True)
    mode = str(strike_mode).strip().upper()
    if mode == "ABSOLUTE":
        return lo, hi
    if mode != "PCT_SPOT":
        raise ValueError("strike_mode must be 'ABSOLUTE' or 'PCT_SPOT'.")
    if not np.isfinite(spot) or spot <= 0:
        raise ValueError("A positive finite stock price is required for relative strikes.")
    return spot * lo / 100.0, spot * hi / 100.0


def _currency_series(frame, name):
    if name not in frame:
        return pd.Series("", index=frame.index, dtype=object)
    return (frame[name].fillna("").astype(str).str.strip().str.upper()
            .replace({"NAN": "", "NONE": "", "<NA>": ""}))


def rank_options(raw, stocks=None, side="PUT", strike_range=None,
                 strike_mode="PCT_SPOT", stock_price_range=None,
                 expiry=None, dte_range=None, price_basis="LAST",
                 rank_by="PREMIUM", top_n=50, one_per_stock=False,
                 positive_bid_only=False, max_spread_pct=None,
                 currency="USD", allow_unknown_currency=True, today=None):
    """Filter ALREADY LOADED contracts and rank highest values first.

    Use dashboard.screen(...) to fetch a new universe/range from Bloomberg.
    This function makes no requests and cannot recover contracts not in `raw`.

    stocks: exact Bloomberg IDs or short symbols; None means all loaded stocks.
    side: PUT, CALL, or BOTH (P/C are accepted).
    strike_range: inclusive (low, high); (90, 100) with PCT_SPOT means K/S 90%-100%.
    stock_price_range: optional inclusive range of underlying last prices.
    expiry: exact date or ISO date string. dte_range: inclusive calendar days.
    price_basis: LAST, BID, ASK, MID. Missing/invalid prices are never substituted.
    rank_by: PREMIUM, PREMIUM_PCT_SPOT, or PREMIUM_PCT_STRIKE.
    top_n: number of rows returned; None or 0 returns all. Ties use stable ID keys.
    one_per_stock: retain the highest scoring contract per stock AND option side.
    positive_bid_only: additionally require a valid two-sided quote with bid > 0.
    max_spread_pct: optional ceiling for 100 * (ask-bid)/mid; requires valid mid.
    currency: filter known option currencies (default USD); no FX conversion.
    allow_unknown_currency: keep missing currency metadata, explicitly labeled.

    Premium ratios are NOT yields, implied volatility or model-based richness.
    Prices/strikes retain Bloomberg's native units. Contract multipliers and
    adjusted deliverables are not verified. Observation dates do not certify
    intraday freshness. A multi-request snapshot is not simultaneous.
    """
    if not isinstance(raw, pd.DataFrame):
        raise TypeError("raw must be a pandas DataFrame, normally dashboard.raw.")
    side = {"P": "PUT", "C": "CALL"}.get(str(side).upper(), str(side).upper())
    price_basis, rank_by = str(price_basis).upper(), str(rank_by).upper()
    strike_mode = str(strike_mode).upper()
    if side not in ("PUT", "CALL", "BOTH"):
        raise ValueError("side must be PUT, CALL or BOTH.")
    if price_basis not in ("LAST", "BID", "ASK", "MID"):
        raise ValueError("price_basis must be LAST, BID, ASK or MID.")
    if rank_by not in ("PREMIUM", "PREMIUM_PCT_SPOT", "PREMIUM_PCT_STRIKE"):
        raise ValueError("Unknown rank_by: " + rank_by)
    if strike_mode not in ("ABSOLUTE", "PCT_SPOT"):
        raise ValueError("strike_mode must be ABSOLUTE or PCT_SPOT.")
    strikes = checked_range(strike_range, "Strike range", positive=True)
    spots = checked_range(stock_price_range, "Stock-price range", positive=True)
    days = checked_range(dte_range, "DTE range")
    if days is not None and (days[0] < 0 or any(x != int(x) for x in days)):
        raise ValueError("DTE bounds must be nonnegative whole calendar days.")
    if top_n is not None and (isinstance(top_n, bool) or not np.isfinite(float(top_n))
                              or int(top_n) != float(top_n) or int(top_n) < 0):
        raise ValueError("top_n must be a nonnegative integer, or None.")
    if max_spread_pct is not None and (not np.isfinite(float(max_spread_pct))
                                       or float(max_spread_pct) < 0):
        raise ValueError("max_spread_pct must be finite and nonnegative, or None.")
    columns = ["RANK", "SIDE_RANK", "STOCK_SIDE_RANK", "TICKER", "NAME", "SIDE", "ID",
               "STOCK_LAST", "EXPIRY", "DTE", "STRIKE", "STRIKE_PCT_SPOT", "PRICE_BASIS",
               "PREMIUM", "PREMIUM_PCT_SPOT", "PREMIUM_PCT_STRIKE", "RANK_METRIC", "RANK_VALUE",
               "LAST", "BID", "ASK", "MID", "SPREAD_PCT", "PRICE_CURRENCY", "CURRENCY_CHECK",
               "PRICE_OBSERVATION_DATE", "QUOTE_NOTE"]
    if raw.empty:
        return pd.DataFrame(columns=columns)
    required = {"TICKER", "ID", "SIDE", "STRIKE", "STOCK_LAST", "EXPIRY"}
    if not required.issubset(raw.columns):
        raise ValueError("Missing columns: " + ", ".join(sorted(required - set(raw.columns))))
    f = raw.copy()
    f["TICKER"] = f["TICKER"].astype(str)
    if stocks is not None:
        selected = resolve_stock_selection(stocks, f["TICKER"].unique())
        f = f[f["TICKER"].isin(selected)].copy()
    if f["ID"].duplicated().any():
        raise ValueError("Duplicate contract IDs in raw: expected one snapshot row per ID.")
    f["SIDE"] = f["SIDE"].astype(str).str.strip().str.upper().replace({"C": "CALL", "P": "PUT"})
    f["STRIKE"] = pd.to_numeric(f["STRIKE"], errors="coerce")
    f["STOCK_LAST"] = pd.to_numeric(f["STOCK_LAST"], errors="coerce")
    f["EXPIRY"] = pd.to_datetime(f["EXPIRY"], errors="coerce", utc=True).dt.tz_convert(None).dt.normalize()
    valid = (f["EXPIRY"].notna() & np.isfinite(f["STRIKE"]) & f["STRIKE"].gt(0)
             & np.isfinite(f["STOCK_LAST"]) & f["STOCK_LAST"].gt(0)
             & f["SIDE"].isin(["CALL", "PUT"]))
    f = f.loc[valid].copy()
    f = enrich_quotes(f, today=today)
    f["STRIKE_PCT_SPOT"] = 100 * f["STRIKE"] / f["STOCK_LAST"]
    f = f.loc[f["DTE"] >= 0].copy()
    if side != "BOTH":
        f = f.loc[f["SIDE"] == side].copy()
    if spots is not None:
        f = f.loc[f["STOCK_LAST"].between(*spots)].copy()
    if strikes is not None:
        values = f["STRIKE"] if strike_mode == "ABSOLUTE" else f["STRIKE_PCT_SPOT"]
        # Compare strikes, not rounded display percentages, at inclusive boundaries.
        lower = strikes[0] if strike_mode == "ABSOLUTE" else f["STOCK_LAST"] * strikes[0] / 100
        upper = strikes[1] if strike_mode == "ABSOLUTE" else f["STOCK_LAST"] * strikes[1] / 100
        f = f.loc[(f["STRIKE"] >= lower) & (f["STRIKE"] <= upper)].copy()
    if expiry is not None:
        date = pd.Timestamp(expiry)
        if pd.isna(date):
            raise ValueError("expiry must be a valid date.")
        if date.tzinfo is not None:
            date = date.tz_localize(None)
        f = f.loc[f["EXPIRY"] == date.normalize()].copy()
    if days is not None:
        f = f.loc[f["DTE"].between(*days)].copy()
    f["PREMIUM"] = pd.to_numeric(f[price_basis], errors="coerce")
    f["PRICE_BASIS"] = price_basis
    f["SPREAD_PCT"] = (100 * (f["ASK"] - f["BID"]) / f["MID"]).where(f["MID"].gt(0))
    f = f.loc[np.isfinite(f["PREMIUM"]) & f["PREMIUM"].gt(0)].copy()
    if positive_bid_only:
        f = f.loc[f["BID"].gt(0) & f["MID"].notna()].copy()
    if max_spread_pct is not None:
        f = f.loc[f["SPREAD_PCT"].notna() & f["SPREAD_PCT"].le(float(max_spread_pct))].copy()
    if price_basis == "MID":
        bc, ac = _currency_series(f, "BID_CURRENCY"), _currency_series(f, "ASK_CURRENCY")
        pc = bc.where(bc.ne(""), ac)
        complete_price_currency = bc.ne("") & ac.ne("") & bc.eq(ac)
        bd = pd.to_datetime(f["BID_DATE"], errors="coerce", utc=True).dt.normalize()
        ad = pd.to_datetime(f["ASK_DATE"], errors="coerce", utc=True).dt.normalize()
        f["PRICE_OBSERVATION_DATE"] = bd.where(bd.notna() & ad.notna() & bd.eq(ad))
    else:
        pc = _currency_series(f, price_basis + "_CURRENCY")
        complete_price_currency = pc.ne("")
        f["PRICE_OBSERVATION_DATE"] = pd.to_datetime(f[price_basis + "_DATE"], errors="coerce", utc=True)
    sc = _currency_series(f, "STOCK_CURRENCY")
    kc = _currency_series(f, "STRIKE_CURRENCY")
    kc = kc.where(kc.ne(""), sc)
    spot_mismatch = pc.ne("") & sc.ne("") & pc.ne(sc)
    strike_mismatch = pc.ne("") & kc.ne("") & pc.ne(kc)
    f["PRICE_CURRENCY"] = pc.replace("", "UNKNOWN")
    f["CURRENCY_CHECK"] = np.where(complete_price_currency, "Quote currency available", "Quote currency metadata incomplete")
    f["PREMIUM_PCT_SPOT"] = (100 * f["PREMIUM"] / f["STOCK_LAST"]).where(~spot_mismatch)
    f["PREMIUM_PCT_STRIKE"] = (100 * f["PREMIUM"] / f["STRIKE"]).where(~strike_mismatch)
    needed = complete_price_currency.copy()
    if rank_by == "PREMIUM_PCT_SPOT":
        needed &= sc.ne("")
        f.loc[sc.eq(""), "CURRENCY_CHECK"] = "Stock currency metadata incomplete"
    elif rank_by == "PREMIUM_PCT_STRIKE":
        needed &= kc.ne("")
        f.loc[kc.eq(""), "CURRENCY_CHECK"] = "Strike/stock currency metadata incomplete"
    currency_mask = pd.Series(True, index=f.index)
    if currency is not None:
        wanted = str(currency).strip().upper()
        if not wanted:
            raise ValueError("currency must be a currency code, or None.")
        currency_mask = pc.eq(wanted) | (pc.eq("") & allow_unknown_currency)
    if not allow_unknown_currency:
        currency_mask &= needed
    f = f.loc[currency_mask].copy()
    if rank_by == "PREMIUM" and currency is None:
        known = set(f["PRICE_CURRENCY"]) - {"UNKNOWN"}
        if len(known) > 1:
            raise ValueError("Raw premiums span multiple currencies; select one currency before ranking.")
    f["RANK_METRIC"] = rank_by
    f["RANK_VALUE"] = f[rank_by]
    f = f.loc[np.isfinite(f["RANK_VALUE"]) & f["RANK_VALUE"].gt(0)].copy()
    f = f.sort_values(["RANK_VALUE", "TICKER", "EXPIRY", "STRIKE", "SIDE", "ID"],
                      ascending=[False, True, True, True, True, True], kind="mergesort")
    if one_per_stock:
        f = f.drop_duplicates(["TICKER", "SIDE"], keep="first")
    f["RANK"] = np.arange(1, len(f) + 1)
    f["SIDE_RANK"] = f.groupby("SIDE", sort=False).cumcount() + 1
    f["STOCK_SIDE_RANK"] = f.groupby(["TICKER", "SIDE"], sort=False).cumcount() + 1
    eligible = len(f)
    if top_n:
        f = f.head(int(top_n)).copy()
    if "NAME" not in f:
        f["NAME"] = ""
    for column in columns:
        if column not in f:
            f[column] = np.nan
    extra = [c for c in f.columns if c not in columns]
    f = f[columns + extra].reset_index(drop=True)
    f.attrs["ranking_scope"] = "Loaded contracts only; no Bloomberg request was made by rank_options."
    f.attrs["eligible_rows_before_top_n"] = eligible
    f.attrs["price_units"] = "Native Bloomberg price units; no contract multiplier or FX conversion."
    return f

## 5 · Interactive dashboard and direct-query function
Query controls require Load / rebuild. Side, premium basis, metric and Top N rerank locally. Highest per stock / side means one contract per stock for PUT or CALL, or up to two for BOTH.

In [ ]:
# 4. Notebook dashboard (ipywidgets only; no Dash/Streamlit server required)
class OptionsDashboard:
    def __init__(self):
        self.service = bql.Service()
        self.members = pd.DataFrame()
        self.metadata = pd.DataFrame()
        self.raw = pd.DataFrame()
        self.paired = pd.DataFrame()
        self.coverage = pd.DataFrame()
        self.errors = pd.DataFrame()
        self.loaded_config = None
        self.last_load_ok = False
        self.last_load_error = None
        self._suspend_updates = False
        self._busy = False
        self._rendering = False
        wide = widgets.Layout(width="310px")
        style = {"description_width": "initial"}
        self.mode = widgets.ToggleButtons(options=["Single stock", "Selected stocks", "All S&P 500"], description="Universe:")
        self.stock = widgets.Combobox(options=[], value=DEFAULT_STOCK, ensure_option=True,
                                      description="Stock:", layout=wide, style=style)
        self.selected_stocks = widgets.Textarea(value="AAPL, MSFT, NVDA", description="Stocks:",
            placeholder="AAPL, MSFT, NVDA or comma-separated full Bloomberg IDs", style=style,
            disabled=True, layout=widgets.Layout(width="640px", height="65px"))
        self.stock_range_on = widgets.Checkbox(value=False, description="Filter stock last-price range", indent=False)
        self.stock_min = widgets.FloatText(value=50, description="Stock min:", style=style, layout=wide, disabled=True)
        self.stock_max = widgets.FloatText(value=500, description="Stock max:", style=style, layout=wide, disabled=True)
        self.expiry_mode = widgets.Dropdown(options=["Nearest target DTE", "Exact date", "All expiries in window", "DTE range"],
                                            description="Expiry selection:", layout=wide, style=style)
        self.target = widgets.BoundedIntText(value=30, min=0, max=1095, description="Target DTE:", style=style,
                                             layout=widgets.Layout(width="200px"))
        self.window = widgets.BoundedIntText(value=15, min=0, max=365, description="Window +/- days:", style=style,
                                             layout=widgets.Layout(width="210px"))
        self.exact = widgets.DatePicker(value=market_date() + timedelta(days=30), description="Exact expiry:",
                                        disabled=True, style=style, layout=wide)
        self.dte_min = widgets.BoundedIntText(value=20, min=0, max=3650, description="DTE min:", style=style,
                                              layout=widgets.Layout(width="200px"), disabled=True)
        self.dte_max = widgets.BoundedIntText(value=45, min=0, max=3650, description="DTE max:", style=style,
                                              layout=widgets.Layout(width="200px"), disabled=True)
        self.strike_mode = widgets.Dropdown(options=[("Strike as % of spot", "PCT_SPOT"), ("Absolute strike", "ABSOLUTE")],
                                            value="PCT_SPOT", description="Strike mode:", style=style, layout=wide)
        self.strike_min = widgets.FloatText(value=80, description="Strike min:", style=style, layout=widgets.Layout(width="200px"))
        self.strike_max = widgets.FloatText(value=120, description="Strike max:", style=style, layout=widgets.Layout(width="200px"))
        self.strikes = widgets.Dropdown(options=[("All within range", 0), ("5 nearest spot", 5), ("11 nearest spot", 11), ("21 nearest spot", 21)], value=0,
                                        description="Strikes / expiry:", style=style, layout=wide)
        self.load_btn = widgets.Button(description="Load / rebuild", button_style="primary", icon="refresh")
        self.quote_btn = widgets.Button(description="Refresh quotes only", disabled=True)
        self.progress = widgets.IntProgress(value=0, min=0, max=1, description="Progress:", layout=widgets.Layout(width="420px"))
        self.status = widgets.HTML("Click Load / rebuild to request Bloomberg data.")
        self.snapshot = widgets.HTML("")
        self.view = widgets.Dropdown(options=["Premium ranking", "Call / put pairs", "Individual contracts", "Stock coverage", "Errors"],
                                     description="Table:", style=style, layout=wide)
        self.price = widgets.Dropdown(options=[("Last", "LAST"), ("Mid (calculated)", "MID"), ("Bid", "BID"), ("Ask", "ASK")],
                                      value="LAST", description="Price shown:", style=style, layout=wide)
        self.rank_side = widgets.Dropdown(options=[("Puts", "PUT"), ("Calls", "CALL"), ("Both", "BOTH")],
                                           value="PUT", description="Rank side:", style=style, layout=wide)
        self.rank_metric = widgets.Dropdown(options=[("Premium (highest first)", "PREMIUM"),
            ("Premium / stock last (%)", "PREMIUM_PCT_SPOT"), ("Premium / strike (%)", "PREMIUM_PCT_STRIKE")],
            value="PREMIUM", description="Rank by:", style=style, layout=widgets.Layout(width="360px"))
        self.rank_topn = widgets.IntText(value=50, description="Top N (0=all):", style=style, layout=widgets.Layout(width="200px"))
        self.rank_one_per = widgets.Checkbox(value=False, description="Highest per stock / side only", indent=False)
        self.rank_positive_bid = widgets.Checkbox(value=False, description="Require positive bid + valid mid", indent=False)
        self.rank_spread_on = widgets.Checkbox(value=False, description="Limit bid/ask spread", indent=False)
        self.rank_spread_max = widgets.FloatText(value=25, description="Max spread / mid %:", style=style,
                                                layout=wide, disabled=True)
        self.rank_unknown_currency = widgets.Checkbox(value=True, description="Allow missing currency metadata", indent=False)
        self.rank_info = widgets.HTML("")
        self.details = widgets.Checkbox(value=False, description="Show IDs, currencies and observation dates", indent=False,
                                        layout=widgets.Layout(width="370px"))
        self.search = widgets.Text(value="", description="Search:", placeholder="Ticker, company, contract, or error",
                                   continuous_update=False, style=style, layout=widgets.Layout(width="420px"))
        self.page = widgets.BoundedIntText(value=1, min=1, max=1, description="Page:", layout=widgets.Layout(width="145px"))
        self.prev_btn = widgets.Button(description="Previous", layout=widgets.Layout(width="90px"))
        self.next_btn = widgets.Button(description="Next", layout=widgets.Layout(width="75px"))
        self.page_info = widgets.HTML("")
        self.export_btn = widgets.Button(description="Export current table", disabled=True, icon="download")
        self.export_output = widgets.Output()
        self.table = widgets.HTML("")
        header = widgets.HTML("""
        <h2 style="margin-bottom:4px">S&amp;P 500 · Options premium screener</h2>
        <div>Selected stocks, strike ranges and highest-premium put/call rankings | Bloomberg BQL snapshots</div>
        <p><b>Last is Bloomberg PX_LAST, not a guaranteed current executable price.</b>
        Bid/ask may be delayed, unavailable, or stale. Mid is calculated, not a trade.
        Fetch times are not exchange timestamps. No live streaming or automatic polling.</p>
        """)
        help_text = widgets.HTML("""
        <details><summary><b>How to use / interpretation</b></summary>
        <p>Select Single stock, Selected stocks, or All S&amp;P 500. Selected stocks accepts unique
        short symbols or comma-separated full Bloomberg IDs; every stock must be a current returned
        constituent. Optional stock-price bounds filter before chain requests. Missing stock prices
        are not silently filled. Stock-price bounds use native Bloomberg price units.</p>
        <p>Strike mode <b>% of spot</b>: 90 to 100 means K is 90% to 100% of the underlying last.
        <b>Absolute strike</b>: 100 to 200 means listed strikes from 100 through 200, inclusive.
        Leave <b>Strikes / expiry = All within range</b> for a complete ranking inside your bounds.
        A nearest-strike cap can omit higher-premium contracts.</p>
        <p>Nearest target DTE selects the closest available expiry separately for each stock;
        different stocks can have different expiries. Exact date uses one common expiry.
        DTE range includes every returned expiry between the minimum and maximum calendar days
        from New York's current date. Same-day contracts may already have stopped trading.</p>
        <p><b>Load / rebuild</b> refreshes membership, stock observations, bounded option chains,
        and selected quotes. <b>Refresh quotes only</b> keeps the contract universe unchanged;
        the ranking rechecks range bounds against refreshed stock last, but new qualifying contracts
        are NOT discovered until a rebuild. Changing query controls leaves the previous loaded snapshot
        in place until you rebuild. Side, price basis, metric and Top N rerank locally without a request.</p>
        <p>Premium ranks the selected LAST/BID/ASK/MID descending. Premium / stock last and
        premium / strike are simple percentage ratios, not yields, IV or a fair-value model.
        Highest per stock / side retains one actual contract per stock and side. BOTH can show two
        entries per stock. Top N is an overall row cap, not a cap per side. Ties are resolved by ticker,
        expiry, strike, side and contract ID. Search operates within the returned Top N.</p>
        <p>Zero, negative, missing and nonfinite selected premiums are excluded from the ranking.
        No last/mid substitution occurs. Mid is blank for invalid, crossed, currency-mismatched or
        observation-date-mismatched bid/ask. Positive-bid and spread filters require a valid mid.
        Spread % is 100 × (ask − bid) / mid. These filters do not verify execution or liquidity.</p>
        <p>Rankings filter known quote currencies to USD. Missing currency metadata is retained and
        labeled by default; uncheck the option to exclude it. No FX conversion or contract multiplier
        is applied. Adjusted deliverables are NOT verified; inspect contract IDs. Raw premium is not
        a valuation verdict. Compare maturity, moneyness and contract terms before interpreting.</p>
        <p>Bloomberg observation dates are not certified intraday quote timestamps; retrieval time
        is not market time. Quotes can be delayed, stale or unavailable. A multi-request scan is not
        simultaneous. Coverage and Errors expose missing chains and failed requests; rankings are only
        among successfully loaded contracts. Paired quotes never average ambiguous series.</p>
        <p>Export writes every row in the current filtered/Top-N table, not just the current page,
        to bquant_exports. Set Top N to 0 to export all eligible ranked contracts.
        Follow Bloomberg and exchange data-use terms. Nothing here places trades.</p>
        </details>
        """)
        box = lambda children: widgets.HBox(children, layout=widgets.Layout(flex_flow="row wrap"))
        self.rank_controls = widgets.VBox([
            box([self.rank_side, self.rank_metric, self.rank_topn, self.rank_one_per]),
            box([self.rank_positive_bid, self.rank_spread_on, self.rank_spread_max]),
            box([self.rank_unknown_currency]), self.rank_info,
        ])
        self.widget = widgets.VBox([
            header, self.mode, box([self.stock, self.selected_stocks]),
            box([self.stock_range_on, self.stock_min, self.stock_max]),
            box([self.expiry_mode, self.exact, self.dte_min, self.dte_max]),
            box([self.target, self.window]),
            box([self.strike_mode, self.strike_min, self.strike_max, self.strikes]),
            box([self.load_btn, self.quote_btn, self.progress]), self.status, self.snapshot,
            box([self.view, self.price, self.details]), self.rank_controls,
            box([self.search, self.export_btn]),
            box([self.prev_btn, self.page, self.next_btn, self.page_info]),
            self.table, self.export_output, help_text,
        ])
        self._query_controls = [self.mode, self.stock, self.selected_stocks, self.stock_range_on,
            self.stock_min, self.stock_max, self.expiry_mode, self.target, self.window, self.exact,
            self.dte_min, self.dte_max, self.strike_mode, self.strike_min, self.strike_max, self.strikes]
        self._ranking_controls = [self.rank_side, self.rank_metric, self.rank_topn, self.rank_one_per,
            self.rank_positive_bid, self.rank_spread_on, self.rank_spread_max, self.rank_unknown_currency]
        for control in self._query_controls:
            control.observe(self._filter_changed, names="value")
        for control in [self.view, self.price, self.details, self.search] + self._ranking_controls:
            control.observe(self._display_changed, names="value")
        self.page.observe(lambda change: self.render(), names="value")
        self.prev_btn.on_click(lambda _: self._turn_page(-1))
        self.next_btn.on_click(lambda _: self._turn_page(1))
        self.load_btn.on_click(self.load)
        self.quote_btn.on_click(self.refresh_quotes)
        self.export_btn.on_click(self.export)
        self._sync_controls()

    def _message(self, text, error=False):
        self.status.value = ("<b>" + ("Error: " if error else "") + html.escape(str(text)) + "</b>")

    def _filter_changed(self, change=None):
        if self._busy or self._suspend_updates:
            return
        self._sync_controls()
        if self.loaded_config is not None:
            self._message("Query filters changed. Displayed prices/rankings use the PREVIOUS loaded selection. Click Load / rebuild to apply.")

    def _sync_controls(self):
        self.stock.disabled = self._busy or self.mode.value != "Single stock"
        self.selected_stocks.disabled = self._busy or self.mode.value != "Selected stocks"
        self.stock_min.disabled = self.stock_max.disabled = self._busy or not self.stock_range_on.value
        exact = self.expiry_mode.value == "Exact date"
        dte = self.expiry_mode.value == "DTE range"
        self.exact.disabled = self._busy or not exact
        self.target.disabled = self.window.disabled = self._busy or exact or dte
        self.dte_min.disabled = self.dte_max.disabled = self._busy or not dte
        self.rank_spread_max.disabled = self._busy or not self.rank_spread_on.value
        self.rank_controls.layout.display = "" if self.view.value == "Premium ranking" else "none"

    def _set_busy(self, value):
        self._busy = value
        for c in self._query_controls + self._ranking_controls:
            c.disabled = value
        self.load_btn.disabled = value
        self.quote_btn.disabled = value or self.metadata.empty
        self.export_btn.disabled = value or self.raw.empty
        self._sync_controls()

    def _display_changed(self, change=None):
        if self._rendering or self._suspend_updates:
            return
        self._sync_controls()
        self.page.value = 1
        self.render()

    def _turn_page(self, step):
        self.page.value = min(self.page.max, max(1, self.page.value + step))

    def _config(self):
        if self.expiry_mode.value == "DTE range":
            lo, hi = checked_range((self.dte_min.value, self.dte_max.value), "DTE range")
            today = market_date()
            start, end = today + timedelta(days=int(lo)), today + timedelta(days=int(hi))
            target = today + timedelta(days=int((lo + hi) // 2))
        else:
            start, end, target = date_window(self.expiry_mode.value, self.exact.value,
                                              self.target.value, self.window.value)
        bounds = checked_range((self.strike_min.value, self.strike_max.value), "Strike range", positive=True)
        spots = checked_range((self.stock_min.value, self.stock_max.value), "Stock-price range", positive=True) if self.stock_range_on.value else None
        return {"mode": self.mode.value, "expiry_mode": self.expiry_mode.value,
                "start": start, "end": end, "target": target,
                "strike_range": bounds, "strike_mode": self.strike_mode.value,
                "stock_price_range": spots, "strikes": int(self.strikes.value),
                "stock_list": self.selected_stocks.value}

    def _quote_progress(self, done, total):
        self.progress.max = max(total, 1)
        self.progress.value = done
        self._message("Retrieving selected option quotes: %s / %s contracts" % (done, total))

    def _finish_quotes(self, raw, errors):
        self.raw = enrich_quotes(raw)
        self.paired = pair_contracts(self.raw)
        self.errors = pd.DataFrame(errors, columns=["STAGE", "SCOPE", "FIELD", "ERROR"])
        if not self.coverage.empty:
            for col in ["CONTRACTS", "CONTRACTS_WITH_LAST", "CONTRACTS_WITH_VALID_MID"]:
                self.coverage[col] = 0
            summary = self.raw.groupby("TICKER").agg(
                CONTRACTS=("ID", "size"), CONTRACTS_WITH_LAST=("LAST", "count"),
                CONTRACTS_WITH_VALID_MID=("MID", "count"))
            for col in summary:
                self.coverage[col] = self.coverage["TICKER"].map(summary[col]).fillna(0).astype(int)
            no_prices = ((self.coverage["CONTRACTS"] > 0) &
                         self.coverage["TICKER"].map(self.raw.groupby("TICKER")[["LAST", "BID", "ASK"]]
                                                    .count().sum(axis=1)).fillna(0).eq(0))
            self.coverage.loc[no_prices, "STATUS"] = "Contracts found; prices unavailable"
            has_prices = (self.coverage["CONTRACTS"] > 0) & ~no_prices
            self.coverage.loc[has_prices, "STATUS"] = "Loaded (quote age unverified)"
        self.page.value = 1
        self.render()

    def _snapshot_label(self):
        c = self.loaded_config
        if c is None:
            return
        strikes = str(c["strikes"]) + " nearest" if c["strikes"] else "all in range"
        tickers = ", ".join(c["tickers"]) if len(c["tickers"]) <= 6 else "%s queried constituents" % len(c["tickers"])
        lo, hi = c["strike_range"]
        units = "% of stock last" if c["strike_mode"] == "PCT_SPOT" else "native strike units"
        self.snapshot.value = ("<p><b>Loaded selection:</b> %s | %s | %s to %s | strikes %g–%g %s | %s / expiry."
            "<br><b>Completed (UTC):</b> %s. This is application time, not quote time. %s stocks excluded by the stock-price range.</p>") % (
            html.escape(tickers), html.escape(c["expiry_mode"]), c["start"], c["end"],
            lo, hi, units, strikes, utc_now(), len(c.get("excluded_by_stock_range", [])))

    def load(self, _=None):
        if self._busy:
            return
        self._set_busy(True)
        self.last_load_ok = False
        self.last_load_error = None
        previous_exists = not self.raw.empty
        previous_state = (self.metadata, self.raw, self.paired, self.coverage, self.errors, self.loaded_config)
        try:
            config = self._config()
            self._message("Loading current S&P 500 constituents and stock-price observations...")
            members = load_members(self.service)
            self.members = members
            tickers = members.index.tolist()
            current = self.stock.value.strip() or DEFAULT_STOCK
            self.stock.options = tickers
            if config["mode"] == "Single stock":
                selected = resolve_stock_selection(current, tickers)
                self.stock.value = selected[0]
            elif config["mode"] == "Selected stocks":
                selected = resolve_stock_selection(config["stock_list"], tickers)
            else:
                selected = tickers
            requested = list(selected)
            if config["stock_price_range"] is not None:
                lo, hi = config["stock_price_range"]
                selected = [t for t in selected if np.isfinite(members.loc[t, "STOCK_LAST"])
                            and lo <= members.loc[t, "STOCK_LAST"] <= hi]
            config["excluded_by_stock_range"] = [t for t in requested if t not in selected]
            config["tickers"] = selected
            self.progress.max = max(len(selected), 1)
            self.progress.value = 0
            self._message("Discovering options within the expiry and strike bounds...")
            blocks, errors = [], []
            coverage = [{"TICKER": t, "NAME": members.loc[t, "NAME"],
                         "STATUS": "Excluded by stock-price range", "DETAIL": "Outside range or stock price unavailable"}
                        for t in config["excluded_by_stock_range"]]
            workers = max(1, min(4, int(MAX_WORKERS)))
            # Bounded waves avoid queueing an entire index after a systemic access failure.
            wave_size = max(1, workers * 4)
            metadata_failures = 0
            stopped = False
            with ThreadPoolExecutor(max_workers=workers) as pool:
                for offset in range(0, len(selected), wave_size):
                    wave = selected[offset:offset + wave_size]
                    if workers == 1:
                        results = (discover_stock(ticker, members.loc[ticker].to_dict(), config) for ticker in wave)
                    else:
                        futures = [pool.submit(discover_stock, ticker, members.loc[ticker].to_dict(), config) for ticker in wave]
                        results = (future.result() for future in futures)
                    for ticker, result in zip(wave, results):
                        frame, status, error = result
                        coverage.append({"TICKER": ticker, "NAME": members.loc[ticker, "NAME"],
                                         "STATUS": status, "DETAIL": error})
                        if not frame.empty:
                            blocks.append(frame)
                        if error:
                            errors.append({"STAGE": "Metadata", "SCOPE": ticker, "FIELD": "Chain",
                                           "ERROR": error})
                        metadata_failures += int(status == "Metadata error")
                        self.progress.value += 1
                        self._message("Option-chain discovery: %s / %s stocks" % (self.progress.value, len(selected)))
                    if self.progress.value >= 8 and metadata_failures == self.progress.value:
                        for ticker in selected[offset + len(wave):]:
                            coverage.append({"TICKER": ticker, "NAME": members.loc[ticker, "NAME"],
                                             "STATUS": "Not attempted: repeated metadata failures", "DETAIL": "Resolve the errors, then rebuild."})
                        stopped = True
                        break
            metadata = pd.concat(blocks, ignore_index=True) if blocks else pd.DataFrame()
            self.coverage = pd.DataFrame(coverage)
            self.loaded_config = config
            self.metadata = metadata
            if metadata.empty:
                self.raw, self.paired = pd.DataFrame(), pd.DataFrame()
                self.errors = pd.DataFrame(errors, columns=["STAGE", "SCOPE", "FIELD", "ERROR"])
                self.view.value = "Stock coverage"
                self.render()
                self._snapshot_label()
                self.last_load_ok = True
                self._message("No contracts selected. Review coverage/errors, stock prices and expiry/strike bounds.")
                return
            raw, quote_errors = fetch_quotes(self.service, metadata, self._quote_progress)
            self._finish_quotes(raw, errors + quote_errors)
            if self.view.value in ("Errors", "Stock coverage") and not errors:
                self.view.value = "Premium ranking"
            self._snapshot_label()
            self.last_load_ok = True
            represented = self.raw["TICKER"].nunique()
            self._message("Loaded %s contracts for %s / %s requested stocks. %s diagnostic messages.%s" % (
                len(self.raw), represented, len(selected), len(self.errors),
                " Scan stopped after repeated failures; see coverage." if stopped else ""))
        except Exception as exc:
            self.last_load_error = "%s: %s" % (type(exc).__name__, exc)
            self.metadata, self.raw, self.paired, self.coverage, self.errors, self.loaded_config = previous_state
            self.render()
            # Do not label retained data with a new successful snapshot time.
            self._message("%s: %s%s" % (type(exc).__name__, exc,
                          " Previous displayed prices have not been refreshed." if previous_exists else ""), error=True)
            traceback.print_exc()
        finally:
            self._set_busy(False)

    def refresh_quotes(self, _=None):
        if self._busy or self.metadata.empty:
            return
        self._set_busy(True)
        previous_state = (self.metadata, self.raw, self.paired, self.coverage.copy(), self.errors)
        try:
            self._message("Refreshing stock observations for the loaded contract selection...")
            meta = self.metadata.copy()
            tickers = meta["TICKER"].unique().tolist()
            stock_quotes = execute_snapshot(self.service, tickers, data_items(self.service, ["STOCK_LAST"]))
            stock_quotes["STOCK_FETCHED_UTC"] = utc_now()
            meta["STOCK_LAST"] = pd.to_numeric(meta["TICKER"].map(stock_quotes["STOCK_LAST"]), errors="coerce")
            meta["STOCK_LAST_DATE"] = (meta["TICKER"].map(stock_quotes["STOCK_LAST_DATE"])
                                       if "STOCK_LAST_DATE" in stock_quotes else pd.NaT)
            meta["STOCK_FETCHED_UTC"] = meta["TICKER"].map(stock_quotes["STOCK_FETCHED_UTC"])
            meta["STOCK_CURRENCY"] = (meta["TICKER"].map(stock_quotes["STOCK_LAST_CURRENCY"])
                                      if "STOCK_LAST_CURRENCY" in stock_quotes else "")
            raw, errors = fetch_quotes(self.service, meta, self._quote_progress)
            existing = self.errors[self.errors["STAGE"] == "Metadata"].to_dict("records") if not self.errors.empty else []
            self.metadata = meta
            self._finish_quotes(raw, existing + errors)
            self._snapshot_label()
            self._message("Refreshed the loaded contract universe. Ranking bounds rechecked against updated stock last; rebuild to discover newly qualifying contracts. %s diagnostics." % len(self.errors))
        except Exception as exc:
            self.metadata, self.raw, self.paired, self.coverage, self.errors = previous_state
            self.render()
            self._message("Refresh failed; previous displayed prices retained. %s: %s" % (type(exc).__name__, exc), error=True)
            traceback.print_exc()
        finally:
            self._set_busy(False)

    @property
    def ranked(self):
        """Full Top-N ranked DataFrame, with metadata; reflects the LOADED query bounds."""
        c = self.loaded_config or {}
        return rank_options(self.raw, side=self.rank_side.value,
            strike_range=c.get("strike_range"), strike_mode=c.get("strike_mode", "PCT_SPOT"),
            stock_price_range=c.get("stock_price_range"), price_basis=self.price.value,
            rank_by=self.rank_metric.value, top_n=self.rank_topn.value,
            one_per_stock=self.rank_one_per.value, positive_bid_only=self.rank_positive_bid.value,
            max_spread_pct=self.rank_spread_max.value if self.rank_spread_on.value else None,
            currency="USD", allow_unknown_currency=self.rank_unknown_currency.value)

    def screen(self, stocks=None, strike_range=(90, 110), strike_mode="PCT_SPOT",
               side="PUT", price_basis="LAST", rank_by="PREMIUM", expiry=None,
               dte_range=(20, 45), stock_price_range=None, top_n=50,
               one_per_stock=False, positive_bid_only=False, max_spread_pct=None,
               allow_unknown_currency=True):
        """Load Bloomberg data, update the dashboard and return the ranked DataFrame.

        stocks=None or 'SPX' explicitly scans all returned S&P 500 constituents.
        Otherwise pass a ticker list. All listed strikes inside the supplied range
        are requested, not an ATM-strike sample. expiry overrides dte_range.
        Use dashboard.coverage and dashboard.errors for partial retrievals.
        """
        if self._busy:
            raise RuntimeError("A request is already running in this dashboard.")
        # Validate ranking inputs before changing controls or making a request.
        rank_options(pd.DataFrame(), side=side, strike_range=strike_range, strike_mode=strike_mode,
                     stock_price_range=stock_price_range, dte_range=dte_range if expiry is None else None,
                     price_basis=price_basis, rank_by=rank_by, top_n=top_n, max_spread_pct=max_spread_pct)
        bounds = checked_range(strike_range, "Strike range", positive=True)
        if bounds is None:
            raise ValueError("screen requires an explicit strike_range to bound Bloomberg requests.")
        days = checked_range(dte_range, "DTE range") if expiry is None else None
        if expiry is None and days is None:
            raise ValueError("Provide expiry or an explicit dte_range.")
        if days is not None and days[1] > 3650:
            raise ValueError("DTE maximum must be <= 3650 in this dashboard.")
        expiration = pd.Timestamp(expiry).date() if expiry is not None else None
        if expiration is not None and (pd.isna(expiration) or expiration < market_date()):
            raise ValueError("expiry must be a valid date on or after the New York calendar date.")
        self._suspend_updates = True
        try:
            is_all = stocks is None or (isinstance(stocks, str) and stocks.strip().upper() in ("SPX", "SPX INDEX", "ALL", "ALL S&P 500"))
            self.mode.value = "All S&P 500" if is_all else "Selected stocks"
            if not is_all:
                self.selected_stocks.value = stocks if isinstance(stocks, str) else ", ".join(str(x) for x in stocks)
            self.strike_mode.value = str(strike_mode).upper()
            self.strike_min.value, self.strike_max.value = bounds
            self.strikes.value = 0
            self.stock_range_on.value = stock_price_range is not None
            if stock_price_range is not None:
                self.stock_min.value, self.stock_max.value = checked_range(stock_price_range, "Stock-price range", positive=True)
            if expiration is not None:
                self.expiry_mode.value = "Exact date"
                self.exact.value = expiration
            else:
                self.expiry_mode.value = "DTE range"
                self.dte_min.value, self.dte_max.value = (int(days[0]), int(days[1]))
            self.rank_side.value = {"P": "PUT", "C": "CALL"}.get(str(side).upper(), str(side).upper())
            self.price.value = str(price_basis).upper()
            self.rank_metric.value = str(rank_by).upper()
            self.rank_topn.value = 0 if top_n is None else int(top_n)
            self.rank_one_per.value = bool(one_per_stock)
            self.rank_positive_bid.value = bool(positive_bid_only)
            self.rank_spread_on.value = max_spread_pct is not None
            if max_spread_pct is not None:
                self.rank_spread_max.value = float(max_spread_pct)
            self.rank_unknown_currency.value = bool(allow_unknown_currency)
            self.view.value = "Premium ranking"
            self.search.value = ""
        finally:
            self._suspend_updates = False
            self._sync_controls()
        self.load()
        if not self.last_load_ok:
            raise RuntimeError(self.last_load_error or "Bloomberg request failed; see dashboard status.")
        if not self.errors.empty:
            warnings.warn("Some requests or data checks reported diagnostics. The ranking is among successfully loaded contracts only; inspect dashboard.coverage and dashboard.errors.", RuntimeWarning)
        return self.ranked

    def current_table(self, filtered=True):
        if self.view.value == "Premium ranking":
            f = self.ranked
            if not self.details.value:
                base = ["RANK", "TICKER", "SIDE", "STOCK_LAST", "EXPIRY", "DTE", "STRIKE", "STRIKE_PCT_SPOT",
                        "PRICE_BASIS", "PREMIUM", "PREMIUM_PCT_SPOT", "PREMIUM_PCT_STRIKE", "RANK_METRIC", "RANK_VALUE",
                        "BID", "ASK", "SPREAD_PCT", "PRICE_CURRENCY", "CURRENCY_CHECK", "PRICE_OBSERVATION_DATE",
                        "QUOTES_FETCHED_UTC", "QUOTE_NOTE", "ID"]
                f = f[[c for c in base if c in f]]
        elif self.view.value == "Stock coverage":
            f = self.coverage.copy()
        elif self.view.value == "Errors":
            f = self.errors.copy()
        elif self.view.value == "Individual contracts":
            f = self.raw.copy()
            if not f.empty:
                f["SELECTED_PRICE"] = f[self.price.value]
                f["PRICE_BASIS"] = self.price.value
                base = ["TICKER", "NAME", "STOCK_LAST", "EXPIRY", "DTE", "STRIKE", "STRIKE_VS_SPOT_PCT",
                        "SIDE", "ID", "PRICE_BASIS", "SELECTED_PRICE", "LAST", "BID", "ASK", "MID", "QUOTE_NOTE"]
                extra = [c for c in f.columns if c not in base]
                f = f[base + (extra if self.details.value else [])]
        else:
            f = self.paired.copy()
            if not f.empty:
                f["CALL_PRICE"] = f["CALL_" + self.price.value]
                f["PUT_PRICE"] = f["PUT_" + self.price.value]
                f["PRICE_BASIS"] = self.price.value
                base = ["TICKER", "NAME", "STOCK_LAST", "EXPIRY", "DTE", "STRIKE", "STRIKE_VS_SPOT_PCT",
                        "PRICE_BASIS", "CALL_PRICE", "PUT_PRICE", "CALL_LAST", "PUT_LAST", "CALL_BID", "CALL_ASK", "PUT_BID", "PUT_ASK", "PAIR_NOTE"]
                extra = [c for c in f.columns if c not in base]
                f = f[base + (extra if self.details.value else [])]
        if filtered and not f.empty and self.search.value.strip():
            text = self.search.value.strip()
            mask = pd.Series(False, index=f.index)
            for c in f.select_dtypes(include=["object", "string"]).columns:
                mask |= f[c].fillna("").astype(str).str.contains(text, case=False, regex=False)
            f = f.loc[mask]
        return f

    def render(self):
        if self._rendering:
            return
        self._rendering = True
        try:
            f = self.current_table()
            n = len(f)
            self.rank_info.value = ("<p><b>Highest first:</b> %s · %s · %s. Ranking covers loaded, eligible contracts only. "
                "Known option currencies are restricted to USD; missing metadata is labeled. No FX conversion or multiplier. "
                "A high premium is not a model-based overvaluation signal.</p>" % (
                    html.escape(self.rank_side.label), html.escape(self.rank_metric.label), html.escape(self.price.label)))
            pages = max(1, math.ceil(n / PAGE_SIZE))
            self.page.max = pages
            self.page.value = min(self.page.value, pages)
            self.prev_btn.disabled = self.page.value <= 1
            self.next_btn.disabled = self.page.value >= pages
            start = (self.page.value - 1) * PAGE_SIZE
            self.page_info.value = "%s rows · page %s of %s · price selector: %s" % (n, self.page.value, pages, self.price.label)
            if f.empty:
                self.table.value = "<p>No eligible rows in this view. Check the selected side, ranges and quote filters; review Stock coverage / Errors, or clear Search.</p>"
                return
            shown = f.iloc[start:start + PAGE_SIZE].copy()
            for col in shown.columns:
                if pd.api.types.is_datetime64_any_dtype(shown[col]):
                    shown[col] = shown[col].dt.strftime("%Y-%m-%d").fillna("")
            shown = shown.rename(columns=lambda c: c.replace("_", " "))
            table = shown.to_html(index=False, escape=True, na_rep="—", border=0,
                                  classes="bq-options-table", float_format=lambda value: format(value, ",.3f"))
            self.table.value = """<style>
            .bq-options-wrap {overflow:auto;max-height:620px;border:1px solid #8884;}
            .bq-options-table {border-collapse:collapse;font:12px sans-serif;white-space:nowrap;width:100%;}
            .bq-options-table th {position:sticky;top:0;background:var(--jp-layout-color2,#eee);padding:9px;text-align:right;}
            .bq-options-table td {padding:7px 10px;border-bottom:1px solid #8883;text-align:right;}
            </style><div class="bq-options-wrap">""" + table + "</div>"
        except Exception as exc:
            self.table.value = "<p><b>Ranking/display error:</b> " + html.escape(str(exc)) + "</p>"
            self.page_info.value = "No ranked result: fix the displayed filter error."
        finally:
            self._rendering = False

    def export(self, _=None):
        try:
            table = self.current_table()
            if table.empty:
                self._message("There are no matching rows to export.")
                return
            folder = Path("bquant_exports")
            folder.mkdir(exist_ok=True)
            stamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%S%fZ")
            prefix = "sp500_options_ranking_" if self.view.value == "Premium ranking" else "sp500_options_"
            path = folder / (prefix + stamp + ".csv")
            table.to_csv(path, index=False)
            with self.export_output:
                clear_output(wait=True)
                print("Saved %s rows. Bloomberg/exchange data-use permissions still apply." % len(table))
                display(FileLink(str(path)))
        except Exception as exc:
            self._message("Export failed: " + str(exc), error=True)

## 6 · Launch
Run this cell to create `dashboard`. An initial one-stock request runs unless `AUTO_LOAD_SINGLE=False`. All-stock scans require an explicit action.

In [ ]:
# 5. Launch. No index-wide request is started automatically.
dashboard = OptionsDashboard()
display(dashboard.widget)
if AUTO_LOAD_SINGLE:
    dashboard.load()

# dashboard.ranked: current full Top-N ranking, including metadata
# dashboard.raw: all loaded option contracts and quotes, before ranking filters
# dashboard.paired: side-by-side call/put quotes
# dashboard.coverage / dashboard.errors: completeness and request diagnostics
# dashboard.members: latest returned S&P 500 constituent table
# dashboard.screen(...): make new Bloomberg requests and return a ranked DataFrame
# rank_options(dashboard.raw, ...): rerank only loaded contracts, without requests

## 7 · Direct function examples
Run the next cell unchanged to do nothing. Set **exactly one** example flag to `True`
to make the corresponding new Bloomberg request. These flags prevent Run All from
silently starting multiple or index-wide scans. Every call updates the same dashboard.


In [ ]:
RUN_SELECTED_PUTS = False
RUN_SELECTED_CALLS = False
RUN_ALL_STOCKS = False

if RUN_SELECTED_PUTS:
    puts = dashboard.screen(
        stocks=["AAPL", "MSFT", "NVDA"],
        strike_range=(90, 100), strike_mode="PCT_SPOT",
        dte_range=(20, 45), side="PUT", price_basis="LAST",
        rank_by="PREMIUM", top_n=50,
    )
    display(puts)

if RUN_SELECTED_CALLS:
    calls = dashboard.screen(
        stocks=["AAPL", "MSFT", "NVDA"],
        strike_range=(100, 110), strike_mode="PCT_SPOT",
        dte_range=(20, 45), side="CALL", price_basis="MID",
        rank_by="PREMIUM_PCT_SPOT", top_n=30,
        one_per_stock=True,
    )
    display(calls)

if RUN_ALL_STOCKS:
    # This is an explicit index-wide request; all listed strikes within bounds are used.
    ranked = dashboard.screen(
        stocks=None,
        stock_price_range=(50, 300),
        strike_range=(50, 250), strike_mode="ABSOLUTE",
        dte_range=(20, 45), side="PUT", price_basis="BID",
        rank_by="PREMIUM_PCT_STRIKE", top_n=50,
        one_per_stock=True, positive_bid_only=True,
        max_spread_pct=25,
    )
    display(ranked)


## 8 · Rerank without a Bloomberg request
`dashboard.ranked` returns the current ranking, with full metadata.
`rank_options(...)` below changes only the local calculation; it does not discover
additional contracts or update the dashboard controls. It cannot recover a stock,
expiry or strike that was excluded when the data was loaded.


In [ ]:
# Optional: run after a successful load. This cell makes no Bloomberg requests.
RUN_LOCAL_RERANK = False
if RUN_LOCAL_RERANK:
    result = rank_options(
        dashboard.raw,
        side="CALL", price_basis="ASK",
        rank_by="PREMIUM_PCT_SPOT",
        top_n=25, one_per_stock=True,
    )
    display(result)

# Programmatic diagnostics / export:
# display(dashboard.coverage)
# display(dashboard.errors)
# dashboard.ranked.to_csv("ranked_options.csv", index=False)


## Interpretation and coverage

| Setting | Definition |
|---|---|
| `PREMIUM` | Chosen LAST/BID/ASK/MID, highest numeric premium first. Native Bloomberg units. |
| `PREMIUM_PCT_SPOT` | `100 × selected premium / stock last`. Not a yield or risk-adjusted value. |
| `PREMIUM_PCT_STRIKE` | `100 × selected premium / strike`. Not annualized and not a return forecast. |
| `PCT_SPOT` strike mode | `100 × strike / stock last`. For example, `(90,100)` means 90%–100% of spot. |
| `ABSOLUTE` strike mode | Actual listed strike values in native units; endpoints included. |
| `one_per_stock=True` | Highest-scoring eligible contract for each stock **and side**. |
| `top_n=0` or `None` | All eligible ranked rows. Otherwise a global row cap. |

Raw premium is affected by intrinsic value, time to expiry, volatility and other factors.
A high premium alone does **not** establish that an option is overvalued. Ratios remove
some nominal scale but do not control for expiry, moneyness, dividends or risk. For like-for-like
inspection, use one exact expiry and a narrow relative-strike range, and verify contract terms.

**Data limitations:** option currencies known to be non-USD are excluded from dashboard rankings.
Missing currency metadata is retained, visibly flagged, by default. Uncheck **Allow missing currency
metadata** to exclude it. No currency conversion, contract multiplier or adjusted-deliverable validation
is performed. The ranking is of numeric quoted premium units, not verified total cash per contract.

Missing/zero/negative selected premiums are excluded, never replaced by another basis. MID needs a
valid two-sided quote; LAST can be an older observation. Observation dates are not certified intraday
exchange timestamps. The notebook makes manual snapshot requests, not streaming subscriptions.

A quote-only refresh keeps the prior contract universe. Relative range filters are checked again
against updated stock last, but newly qualifying contracts need a rebuild. Pending query-control edits
do not change the loaded selection. Partial failures can change rankings; inspect **Stock coverage**
and **Errors**. Adjusted/duplicate series keep their full IDs and are never averaged.

**Technical references:**
- [Bloomberg BQuant product and entitled-data environment](https://professional.bloomberg.com/products/bloomberg-terminal/research/bquant/)
- [Jupyter ipywidgets widget documentation](https://ipywidgets.readthedocs.io/en/stable/examples/Widget%20List.html)
- [pandas sort_values documentation](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html)
- [Options Industry Council: options pricing](https://www.optionseducation.org/optionsoverview/options-pricing)

Public documentation does not certify Bloomberg native fields or access in your account. The adapter
uses the same BQL fields as the preceding notebook (`expire_dt`, `strike_px`, `put_call`, `px_last`,
`px_bid`, `px_ask`). Any environment-specific changes should be made in `FIELD_NAMES` and verified
with Bloomberg's in-terminal BQL/BQuant help. This version was not live-tested against Bloomberg.
